In [2]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/personal_transactions.csv")

# --- 1. 只保留支出，去掉非消费类别 ---
df = df[df["Transaction Type"] == "debit"].copy()
df = df[~df["Category"].isin(["Credit Card Payment", "Paycheck"])].copy()

# --- 2. 合并小类 ---
category_map = {
    "Restaurants":          "食饮",
    "Coffee Shops":         "食饮",
    "Alcohol & Bars":       "食饮",
    "Fast Food":            "食饮",
    "Food & Dining":        "食饮",
    "Groceries":            "食饮",
    "Shopping":             "购物",
    "Electronics & Software": "购物",
    "Home Improvement":     "购物",
    "Haircut":              "购物",
    "Gas & Fuel":           "出行",
    "Auto Insurance":       "出行",
    "Utilities":            "居家",
    "Mortgage & Rent":      "居家",
    "Internet":             "居家",
    "Mobile Phone":         "居家",
    "Movies & DVDs":        "娱乐",
    "Music":                "娱乐",
    "Television":           "娱乐",
    "Entertainment":        "娱乐",
}
df["label"] = df["Category"].map(category_map)
print(df["label"].value_counts())

label
食饮    260
居家    126
购物    113
出行     70
娱乐     48
Name: count, dtype: int64


In [3]:
# --- 3. 特征工程 ---
df["Date"] = pd.to_datetime(df["Date"])
df["hour"] = 12                                    # 数据没有时间，固定用中午占位
df["day_of_week"] = df["Date"].dt.dayofweek        # 0=周一, 6=周日
df["is_weekend"] = (df["day_of_week"] >= 5).astype(float)
df["amount_log"] = df["Amount"].apply(lambda x: __import__("math").log1p(x))  # 金额取log，压缩大数值

# Description 处理：提取关键词作为类别特征
# 先看看有哪些独特的 Description
print(f"独特商家数: {df['Description'].nunique()}")
print(df["Description"].value_counts().head(10))

独特商家数: 63
Description
Grocery Store       103
Amazon               59
Hardware Store       34
Starbucks            32
American Tavern      24
Brewing Company      23
Mortgage Payment     21
Gas Company          21
Spotify              21
Phone Company        21
Name: count, dtype: int64


In [4]:
import math

# --- 4. 编码 Description ---
# 63个商家，每个分配一个整数ID
desc_encoder = LabelEncoder()
df["desc_id"] = desc_encoder.fit_transform(df["Description"])

# --- 5. 编码 label ---
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print("类别映射:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i} = {cls}")

# --- 6. 转成 tensor ---
# 数值特征：金额(log)、星期几、是否周末
X_num = torch.tensor(
    df[["amount_log", "day_of_week", "is_weekend"]].values,
    dtype=torch.float32
)
# 商家ID：整数，给 Embedding 层用
X_desc = torch.tensor(df["desc_id"].values, dtype=torch.long)
# 标签
y = torch.tensor(df["label_id"].values, dtype=torch.long)

print(f"\n数据量: {len(y)}")
print(f"X_num shape: {X_num.shape}")
print(f"X_desc shape: {X_desc.shape}")
print(f"y shape: {y.shape}")

# --- 7. 切分训练/验证集 ---
n = len(y)
n_train = int(n * 0.8)

# 先打乱顺序
idx = torch.randperm(n)
X_num  = X_num[idx]
X_desc = X_desc[idx]
y      = y[idx]

X_num_train,  X_num_val  = X_num[:n_train],  X_num[n_train:]
X_desc_train, X_desc_val = X_desc[:n_train], X_desc[n_train:]
y_train,      y_val      = y[:n_train],      y[n_train:]

print(f"\n训练集: {len(y_train)} 条")
print(f"验证集: {len(y_val)} 条")

类别映射:
  0 = 出行
  1 = 娱乐
  2 = 居家
  3 = 购物
  4 = 食饮

数据量: 617
X_num shape: torch.Size([617, 3])
X_desc shape: torch.Size([617])
y shape: torch.Size([617])

训练集: 493 条
验证集: 124 条


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BillAutoencoder(nn.Module):
    def __init__(self, input_dim=11, latent_dim=4):
        super().__init__()
        # Encoder：11 → 8 → latent
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, latent_dim)
        )
        # Decoder：latent → 8 → 11
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_reconstructed = self.decoder(z)
        return x_reconstructed

    def reconstruction_error(self, x):
        x_rec = self.forward(x)
        # 每条数据的重建误差：各维度误差平方的均值
        return F.mse_loss(x_rec, x, reduction="none").mean(dim=1)

In [6]:
# --- 8. 定义模型 ---
class BillClassifier(nn.Module):
    def __init__(self, n_desc, embed_dim, n_num_features, n_classes):
        super().__init__()
        # Embedding层：把商家ID(整数) → 向量
        self.embedding = nn.Embedding(n_desc, embed_dim)
        
        # MLP：吃 embedding + 数值特征，输出分类
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim + n_num_features, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, n_classes)
        )
    
    def forward(self, x_desc, x_num):
        # x_desc: [batch_size]        → embedding后: [batch_size, embed_dim]
        emb = self.embedding(x_desc)
        # 拼接: [batch_size, embed_dim + n_num_features]
        x = torch.cat([emb, x_num], dim=1)
        return self.mlp(x)

model = BillClassifier(
    n_desc=63,        # 商家数量
    embed_dim=8,      # 每个商家用8维向量表示
    n_num_features=3, # amount_log, day_of_week, is_weekend
    n_classes=5       # 5个消费类别
)

print(model)
print(f"\n参数总量: {sum(p.numel() for p in model.parameters())}")

BillClassifier(
  (embedding): Embedding(63, 8)
  (mlp): Sequential(
    (0): Linear(in_features=11, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=5, bias=True)
  )
)

参数总量: 1053


In [ ]:
# 加载模型权重
model_loaded = BillClassifier(
    n_desc=63, embed_dim=8,
    n_num_features=3, n_classes=5
)
model_loaded.load_state_dict(torch.load("../data/bill_classifier.pth"))
model_loaded.eval()
print("模型已加载")

In [ ]:
# 先用上周训练好的 embedding 层提取商家向量
# 重新 load 上周模型，或者直接重新训练一遍分类器
# 这里直接重用上周的 model

model_clf = BillClassifier(
    n_desc=63, embed_dim=8,
    n_num_features=3, n_classes=5
)
criterion_clf = nn.CrossEntropyLoss()
optimizer_clf = torch.optim.Adam(model_clf.parameters(), lr=0.01)

for epoch in range(100):
    model_clf.train()
    logits = model_clf(X_desc_train, X_num_train)
    loss = criterion_clf(logits, y_train)
    optimizer_clf.zero_grad()
    loss.backward()
    optimizer_clf.step()

# 提取全量数据的 embedding，拼上数值特征
model_clf.eval()
with torch.no_grad():
    emb_all = model_clf.embedding(X_desc)       # [617, 8]
    X_all = torch.cat([emb_all, X_num], dim=1)  # [617, 11]

# 切分
X_ae_train = X_all[:int(len(X_all)*0.8)]
X_ae_val   = X_all[int(len(X_all)*0.8):]

# 训练 Autoencoder
ae = BillAutoencoder(input_dim=11, latent_dim=4)
optimizer_ae = torch.optim.Adam(ae.parameters(), lr=0.01)

for epoch in range(150):
    ae.train()
    x_rec = ae(X_ae_train)
    loss = F.mse_loss(x_rec, X_ae_train)

    optimizer_ae.zero_grad()
    loss.backward()
    optimizer_ae.step()

    if epoch % 30 == 0:
        ae.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(ae(X_ae_val), X_ae_val)
        print(f"Epoch {epoch:3d} | train loss: {loss.item():.4f} | val loss: {val_loss.item():.4f}")

In [ ]:
# 计算全量数据的重建误差
ae.eval()
with torch.no_grad():
    errors = ae.reconstruction_error(X_all)

# 用95分位数作为阈值
threshold = torch.quantile(errors, 0.95)
print(f"异常阈值 (95th percentile): {threshold.item():.4f}")

# 找出异常交易
anomaly_mask = errors > threshold
anomaly_idx = anomaly_mask.nonzero().squeeze()

# 还原到原始数据
df_reset = df.reset_index(drop=True)
# 注意：X_all 是打乱过的，需要用 idx 对应回去
df_anomaly = df_reset.iloc[idx[anomaly_idx.tolist()].tolist()]

print(f"\n检测到 {len(df_anomaly)} 条异常交易:")
print(df_anomaly[["Date","Description","Amount","label"]].to_string())